# Quick, Draw!

Can a computer tell what you are drawing? In this notebook you sketch something simple, like a cat, a bicycle or a pizza, and a small neural network tries to name it. It guesses again with every stroke you add, so you can watch it change its mind as you draw.

The network was trained on **Quick, Draw!**, a collection of 50 million doodles made by people playing an online game from Google. Each player was given a word and 20 seconds to draw it. The doodles are quick and messy, which makes this a harder job than reading the characters in [[82AE4382-A4A5-4F18-BEA8-7513CE9C418A|Handwritten Digits]] and [[43F5A1FA-59B9-4F18-9DD9-EF942B123E61|Handwritten Letters and Digits]].

## Loading the model

The trained network is attached to this notebook as a file called `quickdraw.onnx` (click the paperclip icon in the top right to see it). ONNX stands for Open Neural Network eXchange, a popular format for sharing trained models. The file is about 420 KB, which is smaller than most photos.

We load it with the `dnn` (deep neural network) module in OpenCV.

In [ ]:
import cv2

net = cv2.dnn.readNetFromONNX('/attachments/quickdraw.onnx')
print("Neural network loaded")

## The doodles the model knows

Quick, Draw! has 345 different words. This model was trained on **100** of them, using 12,000 doodles of each. Each word is called a **category**, and the model can only ever answer with one of these 100.

The model answers with a category number from 0 to 99. `LABELS` turns that number into a word. Have a look through the list: these are the things you can draw.

In [ ]:
LABELS = [
  "airplane", "alarm clock", "ant", "apple", "banana",
  "baseball bat", "basketball", "bed", "bee", "bicycle",
  "bird", "book", "bowtie", "bridge", "broom",
  "bus", "butterfly", "cactus", "cake", "camera",
  "candle", "car", "carrot", "castle", "cat",
  "chair", "clock", "cloud", "crab", "crown",
  "cup", "diamond", "dog", "donut", "door",
  "drums", "duck", "ear", "elephant", "envelope",
  "eye", "eyeglasses", "fish", "flower", "fork",
  "frog", "giraffe", "guitar", "hamburger", "hammer",
  "hand", "hat", "headphones", "helicopter", "hot air balloon",
  "house", "ice cream", "key", "ladder", "light bulb",
  "lightning", "lollipop", "moon", "mountain", "mushroom",
  "octopus", "palm tree", "pants", "paper clip", "pencil",
  "penguin", "piano", "pineapple", "pizza", "rabbit",
  "rainbow", "sailboat", "saw", "scissors", "shark",
  "shoe", "skull", "smiley face", "snail", "snake",
  "snowman", "sock", "spider", "star", "strawberry",
  "sun", "sword", "t-shirt", "table", "tent",
  "tree", "umbrella", "whale", "windmill", "zigzag",
]

print(len(LABELS), "categories")
print("Category 24 is", LABELS[24])

## Matching the line thickness

Every Quick, Draw! doodle was saved with the same thin line, about 1/18 of the size of the whole doodle. The model has only ever seen lines like that. If your lines are much thicker or thinner, a cat stops looking like the cats it learned from.

Your pen has a fixed width, so a small doodle has lines that are thick for its size, and a large doodle has thin ones. The `set_line_thickness` function fixes that. It measures how thick the lines in an image are, then shaves them down or fattens them up until they are the thickness we ask for.

In [ ]:
import numpy as np

def set_line_thickness(image, thickness):
  """Make the lines in an image (0 = empty, 255 = ink) the given
  thickness in pixels."""
  # How far is each ink pixel from the edge of its line? In the middle of
  # a line, that distance is half of the line's thickness.
  depth = cv2.distanceTransform((image > 127).astype(np.uint8), cv2.DIST_L2, 3)
  middles = depth[(depth > 0) & (depth == cv2.dilate(depth, np.ones((3, 3), np.uint8)))]
  if len(middles) == 0:
    return image
  change = round(thickness - 2 * np.median(middles))

  # Dilating makes lines thicker, eroding makes them thinner.
  brush = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (abs(change) + 1, abs(change) + 1))
  return cv2.dilate(image, brush) if change > 0 else cv2.erode(image, brush)

print("set_line_thickness function created")

## Getting your drawing ready for the model

The model can't look at your drawing directly. Every Quick, Draw! image it was trained on is a tiny **28 x 28** grid of pixels, with a white doodle on a black background, so that is the only kind of picture the model understands.

The doodles were also tidied up in the same way: each one was cut out, placed in the middle of a square so that it nearly fills it, and then shrunk to 28 x 28. The `prepare` function below does the same to your drawing, and matches the line thickness on the way. That is why it doesn't matter whether you draw big or small, or in a corner of the pad.

In [ ]:
def prepare(drawing):
  """Turn the drawing (a 2D array, 0 = empty, 255 = ink) into the 28x28
  image the model expects. Returns None if nothing has been drawn."""
  # 1. Find the part of the pad that has ink on it, and cut it out.
  rows, columns = np.nonzero(drawing)
  if len(rows) == 0:
    return None
  doodle = drawing[rows.min():rows.max() + 1, columns.min():columns.max() + 1]

  # 2. Place it in the middle of a black square, with a thin margin
  #    around it, like the Quick, Draw! doodles.
  height, width = doodle.shape
  size = max(height, width)
  margin = size // 26
  square = np.zeros((size + 2 * margin, size + 2 * margin), drawing.dtype)
  top = margin + (size - height) // 2
  left = margin + (size - width) // 2
  square[top:top + height, left:left + width] = doodle

  # 3. Match the line thickness, then shrink the square to 28x28.
  square = set_line_thickness(square, size / 18)
  return cv2.resize(square, (28, 28), interpolation=cv2.INTER_AREA)

print("prepare function created")

## Asking the model

Now for the neural network itself. We hand it the 28 x 28 image, and it gives back 100 scores, one for each category. The higher a score, the more the image looks like that category to the model.

Raw scores are hard to read, so the last two lines turn them into **confidences** that add up to 100%. This step is called *softmax*.

In [ ]:
def predict(image):
  """Run the model on a 28x28 image. Returns 100 confidences (0 to 1),
  one for each category in LABELS."""
  blob = cv2.dnn.blobFromImage(image, scalefactor=1 / 255.0)
  net.setInput(blob)
  scores = net.forward()[0]
  exponents = np.exp(scores - scores.max())
  return exponents / exponents.sum()

print("predict function created")

In [ ]:
PAD = (10, 10, 320, 320)          # where you draw: x, y, width, height
CLEAR_BUTTON = (10, 340, 320, 30)
INK = "#201E1D"
GREEN = "#56BC6C"
GREY = "#C9C5C1"

def show(canvas, image, confidences):
  """Redraw the pad's background and everything to the right of it."""
  ctx = canvas.get_context("2d")
  ctx.clear_rect(0, 0, 600, 380)
  ctx.text_align = "center"

  # The pad and the Clear button. Your strokes sit on a layer above this.
  ctx.fill_style = INK
  ctx.fill_rect(*PAD)
  ctx.fill_rect(*CLEAR_BUTTON)
  ctx.fill_style = "#ffffff"
  ctx.font = "bold 14px sans-serif"
  ctx.fill_text("Clear", 170, 360)
  if image is None:
    ctx.fill_style = "#8a8580"
    ctx.font = "18px sans-serif"
    ctx.fill_text("Draw a doodle here", 170, 175)

  # What the model sees: the 28x28 image, each pixel drawn as a 5x5 block.
  ctx.fill_style = INK
  ctx.font = "12px sans-serif"
  ctx.text_align = "left"
  ctx.fill_text("What the model sees", 350, 22)
  ctx.fill_text("Prediction", 504, 22)
  ctx.fill_text("Top 5 guesses", 350, 196)
  canvas.draw_pixels(np.zeros((28, 28)) if image is None else image, 350, 30, 140, 140)

  # The five most likely categories, best first. The first is the prediction,
  # written one word per line.
  top = [] if confidences is None else [int(i) for i in confidences.argsort()[::-1][:5]]
  words = LABELS[top[0]].split(" ") if top else ["?"]
  ctx.font = "bold 17px sans-serif"
  for line, word in enumerate(words):
    ctx.fill_text(word, 504, 105 - 11 * (len(words) - 1) + 22 * line)

  # One bar per guess.
  for row, label in enumerate(top):
    y = 206 + row * 33
    confidence = float(confidences[label])
    ctx.fill_style = INK
    ctx.font = "bold 14px sans-serif"
    ctx.text_align = "left"
    ctx.fill_text(LABELS[label], 350, y + 18)
    ctx.font = "13px sans-serif"
    ctx.text_align = "right"
    ctx.fill_text(f"{confidence:.0%}", 500, y + 18)
    ctx.fill_style = GREEN if row == 0 else GREY
    ctx.fill_rect(508, y + 5, 82 * confidence, 16)

def is_clear_button(x, y):
  left, top, width, height = CLEAR_BUTTON
  return left <= x <= left + width and top <= y <= top + height

## Draw a doodle!

Time to put it all together. `canvas.on_draw` lets you draw on the pad, and calls our `update` function every time your drawing changes. Each update follows the same three steps: **prepare** the drawing, **predict** the category, and **show** the results.

With 100 categories there isn't room for a bar each, so the canvas shows the model's **top 5 guesses**. Keep an eye on the runners-up: they tell you what else your doodle looks like.

Run the cell, pick something from the `LABELS` list, and draw it on the black pad. Draw it quickly and simply, the way the game's players did. Click **Clear** to start again, and press the Stop button when you are done.

In [ ]:
from codetto import graphics2d

canvas = graphics2d.canvas(600, 380, scale=True)

def update():
  image = prepare(canvas.get_drawing())
  confidences = None if image is None else predict(image)
  show(canvas, image, confidences)
  # The show function is hidden - edit the notebook if you'd like to see it

def handle_click(x, y):
  if is_clear_button(x, y):
    canvas.clear_drawing()
    update()

canvas.on_draw(update, line_width=8, color="#ffffff", area=PAD)
canvas.on_click(handle_click)

update()
canvas.run()

## Things to try

- Draw a circle and look at the guesses. Now add short lines all the way around it to make a **sun**. How many lines does it take before the model agrees?
- Draw a **cat**, starting with the head. When does the model first guess it: after the ears, or the whiskers?
- Draw a curved shape like a crescent. Is it a **moon** or a **banana**? Look at how the confidence is shared when two categories look alike.
- Draw a small doodle in one corner of the pad. The model still sees it large, centered, and with thin lines. Which two functions are responsible for that?
- Draw something that isn't on the list, like a horse or a tornado. The model always picks one of its 100 categories, because they are all it knows. How confident is it?
- Draw something very carefully, with lots of detail. Does the model do better or worse than with a quick sketch? Remember what kind of drawings it learned from.
- In the `prepare` function, put a `#` in front of the `set_line_thickness` line and run that cell and the last cell again. Draw a large doodle, then a small one. What changes in the 28 x 28 image?

The model in this notebook was trained on the [Quick, Draw! Dataset](https://github.com/googlecreativelab/quickdraw-dataset), made available by Google, Inc. under the [Creative Commons Attribution 4.0 International](https://creativecommons.org/licenses/by/4.0/) license.

# Check Your Understanding